In [1]:
import os
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from time import time
import shutil
from tqdm import tqdm

In [2]:
df_train = pd.read_csv('../data/gldv2_metadata/train.csv',dtype={'id': str})
df_train.head(3)

,id,url,landmark_id
0,6e158a47eb2ca3f6,https://upload.wikimedia.org/wikipedia/commons...,142820
1,202cd79556f30760,http://upload.wikimedia.org/wikipedia/commons/...,104169
2,3ad87684c99c06e1,http://upload.wikimedia.org/wikipedia/commons/...,37914


In [3]:
df_train.shape

(4132914, 3)

In [5]:
train_csv_path = Path("../data/gldv2_metadata/train.csv")
dataset_image_root = Path("../data/images_north_america/images")
target_image_dir = Path("../data/my_data/images")
target_csv_path = Path("../data/my_data/train_filtered.csv")

target_image_dir.mkdir(parents=True, exist_ok=True)

print('Reading data....')
train_df = pd.read_csv(train_csv_path)
id_col = 'id'
train_df[id_col] = train_df[id_col].astype(str)

print('Building id -> row index....')
train_df_indexed = train_df.set_index(id_col)
train_id_set = set(train_df_indexed.index)

print('Reading image data....')
image_files = []
image_files.extend(dataset_image_root.rglob('*.jpg'))

matched_ids = []

for img_path in tqdm(image_files, desc='Checking images'):
    img_id = img_path.stem

    if img_id in train_id_set:
        dst_path = target_image_dir / img_path.name
        shutil.copy2(img_path, dst_path)
        matched_ids.append(img_id)
        
print('Creating new DataFrame....')
new_df = train_df_indexed.loc[matched_ids].reset_index()
new_df.to_csv(target_csv_path, index=False)

print('Done.')
print(f"matched images: {len(matched_ids)}")
print(f"new dataframe shape: {new_df.shape}")

Reading data....
Building id -> row index....
Reading image data....


Checking images: 100%|██████████| 395083/395083 [2:44:50<00:00, 39.95it/s]   


Creating new DataFrame....
Done.
matched images: 395083
new dataframe shape: (395083, 3)
